Realizamos las importaciones necesarias:

In [1]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from joblib import parallel_backend

Fijamos las variables globales del cuaderno:

In [2]:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "etiquetas_test_sin_alias.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

Definimos funciones de carga de datos, tokenización, y cálculo de métricas y estadísticas:

In [3]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=',', duplicates=False):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Creamos un set para los títulos ya encontrados a fin de evitar duplicados:
        titles = set()

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            title = row[0].strip().lower()
            summary = row[1].strip().lower()

            if not duplicates:
                if title not in titles:
                    titles.add(title)
                else:
                    continue

            papers.append((title, summary))
            terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    probs = {}
    for label, freq in sorted(ct.items(), key=lambda item: item[1], reverse=True):
        prob = freq / len(texts)
        probs[label] = prob
        print(f"{label}: {freq} ({prob:.1%})", end="  ")
    print("\n")
    return probs


try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

Carga de datos SIN DUPLICADOS:

In [27]:
data = {"nd": {}, "d": {}}

In [28]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, ",", False)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
train_tok, dev_tok, y_train, y_dev = train_test_split(all_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Guardamos los datos sin duplicados en el diccionario 'data':
data["nd"] = {
    "raw_all": raw_all,
    "y_all": y_all,
    "y_bin": y_bin,
    "all_tok": all_tok,
    "train_tok": train_tok,
    "dev_tok": dev_tok,
    "y_train": y_train,
    "y_dev": y_dev
}

Papers de entrenamiento: 32569
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


Carga de datos CON DUPLICADOS:

In [30]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE, duplicates=True)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
all_tok = [f"{title} {tokenizer_s2.sep_token} {summary}" for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
title_counts = collections.Counter([title for title, _ in raw_all])
unique_indices = [i for i, (title, _) in enumerate(raw_all) if title_counts[title] == 1]
dup_indices = [i for i, (title, _) in enumerate(raw_all) if title_counts[title] > 1]

n_dev = int(len(all_tok) * DEV_PART)
test_size_unique = n_dev / len(unique_indices)

train_u_idx, dev_idx = train_test_split(
    unique_indices,
    test_size=test_size_unique,
    shuffle=True,
    random_state=SEED
)

# Unimos los duplicados y los únicos restantes para el conjunto de entrenamiento:
train_idx = dup_indices + train_u_idx

# Extraemos las particiones finales:
train_tok = [all_tok[i] for i in train_idx]
y_train = y_bin[train_idx]

dev_tok = [all_tok[i] for i in dev_idx]
y_dev = y_bin[dev_idx]

# Guardamos los datos sin duplicados en el diccionario 'data':
data["d"] = {
    "raw_all": raw_all,
    "y_all": y_all,
    "y_bin": y_bin,
    "all_tok": all_tok,
    "train_tok": train_tok,
    "dev_tok": dev_tok,
    "y_train": y_train,
    "y_dev": y_dev
}

Papers de entrenamiento: 43555
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


NameError: name 'tokenizer_s2' is not defined

Carga de datos DE TEST:

In [5]:
raw_test, y_test_nobin = load_corpus(TEST_FILE, ";")
print("Papers de test:", len(raw_test))

y_test = mlb.transform(y_test_nobin)

test_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_test]

Papers de test: 4885


## Regresión logística (RL)

Definimos el vectorizador TF-IDF para generar las matrices dispersas de los conjuntos de entrenamiento y validación:

In [8]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=2)
X_all = vec.fit_transform(all_tok)
X_train = vec.transform(train_tok)
X_dev = vec.transform(dev_tok)
X_test = vec.transform(test_tok)

# Ordenamos las entradas de cada fila y sumamos posibles duplicados
# para evitar fallos en la paralelización de la RL:
X_all.sort_indices()
X_all.sum_duplicates()
X_train.sort_indices()
X_train.sum_duplicates()
X_dev.sort_indices()

In [9]:
# Definimos una función para ajustar los umbrales de decisión de un clasificador:
def adjust_thresholds(model, X, y, min_th=0.01, max_th=1, n_th=200, alg="rl"):
    # Obtenemos las probabilidades para cada clase que ha asociado el modelo a cada paper:
    scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Creamos un vector con tantos elementos como clases, inicializado a ceros:
    n_classes = len(mlb.classes_)
    optimal_thresholds = np.zeros(n_classes)

    # Definimos una lista de 'n_th' umbrales de decisión equiespaciados entre 'min_th' y 'max_th':
    thresholds = np.linspace(min_th, max_th, n_th)

    # Iteramos sobre cada clase:
    for c in range(n_classes):
        max_f1_c = -1
        best_th = 0

        # Obtenemos la columna de etiquetas y probabilidades para cada paper asociadas a la clase:
        y_c = y[:, c]
        scores_c = scores[:, c]

        # Iteramos sobre cada umbral:
        for th in thresholds:
            # Obtenemos las predicciones de etiquetas asociadas a ese umbral y calculamos su F1 individual:
            preds_c = (scores_c >= th).astype(int)
            f1_c = f1_score(y_c, preds_c, average="binary", zero_division=0)

            # Si el F1 mejora respecto al que tenía la clase ya, lo guardamos:
            if f1_c > max_f1_c:
                max_f1_c = f1_c
                best_th = th

        # En cada entrada 'i' de 'optimal_thresholds', almacenamos el mejor umbral encontrado para la clase 'c_i':
        optimal_thresholds[c] = best_th

    return optimal_thresholds

In [10]:
# Definimos asimismo una función para obtener las predicciones optimizadas dados los umbrales ajustados:
def predict_with_th(model, X, y, thresholds, verbose=False, alg="rl"):
    # Obtenemos las probabilidades originales:
    scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Actualizamos las predicciones con los umbrales de decisión que hemos calculado:
    preds = (scores >= thresholds).astype(int)

    # Buscamos los papers que se hayan quedado sin clase:
    zero_rows = np.where(preds.sum(axis=1) == 0)[0] # NumPy devuelve una tupla de 1 elemento, por eso lo extraemos

    # Iteramos sobre cada paper sin clase, obtenemos la clase de mayor probabilidad y se la asignamos:
    for r in zero_rows:
        top_class = np.argmax(scores[r])
        preds[r, top_class] = 1

    if verbose:
        # Consultamos las métricas derivadas de usarlos:
        print("=== Métricas con umbrales de decisión optimizados ===")
        print(compute_metrics(y, preds))

    return preds

In [ ]:
for min_df in [1]:
    vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=min_df)
    X_all = vec.fit_transform(all_tok)
    X_train = vec.transform(train_tok)
    X_dev = vec.transform(dev_tok)
    X_test = vec.transform(test_tok)

    # Ordenamos las entradas de cada fila y sumamos posibles duplicados
    # para evitar fallos en la paralelización de la RL:
    X_all.sort_indices()
    X_all.sum_duplicates()
    X_train.sort_indices()
    X_train.sum_duplicates()
    X_dev.sort_indices()

    for c in [0.48, 0.49, 0.51]:
        model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED), n_jobs=12).fit(X_all, y_bin)
        preds = model.predict(X_test)
        print(f"== C={c}, min_df={min_df}")
        print(compute_metrics(y_test, preds))
        print()

== C=0.48, min_df=1
{'accuracy': 0.19201637666325486, 'f1': 0.19470809593632768, 'precision': 0.1740071529914527, 'recall': 0.35735444165307456, 'hamming_loss': 0.05032168445679193}

== C=0.49, min_df=1
{'accuracy': 0.1932446264073695, 'f1': 0.19509852582518433, 'precision': 0.1744979221344709, 'recall': 0.35714313634483236, 'hamming_loss': 0.05019739728030414}

== C=0.51, min_df=1
{'accuracy': 0.1938587512794268, 'f1': 0.19431509941686262, 'precision': 0.1741748082509061, 'recall': 0.3528730418370848, 'hamming_loss': 0.04999268898961837}



In [24]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=1)
X_all = vec.fit_transform(all_tok)
X_train = vec.transform(train_tok)
X_dev = vec.transform(dev_tok)
X_test = vec.transform(test_tok)

# Ordenamos las entradas de cada fila y sumamos posibles duplicados
# para evitar fallos en la paralelización de la RL:
X_all.sort_indices()
X_all.sum_duplicates()
X_train.sort_indices()
X_train.sum_duplicates()
X_dev.sort_indices()

model = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=22.75, class_weight="balanced", random_state=SEED), n_jobs=8).fit(X_all, y_bin)
preds = model.predict(X_test)
print(compute_metrics(y_test, preds))

{'accuracy': 0.2974411463664278, 'f1': 0.16766003331648854, 'precision': 0.1581288381889964, 'recall': 0.21565898925494534, 'hamming_loss': 0.03897499634449481}


In [25]:
model.fit(X_train, y_train)
th = adjust_thresholds(model, X_dev, y_dev)

In [26]:
predict_with_th(model, X_dev, y_dev, th, True)
_ = predict_with_th(model, X_test, y_test, th, True)

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.6469237832874196, 'f1': 0.61281428483878, 'precision': 0.7288470308159607, 'recall': 0.5523891590741833, 'hamming_loss': 0.018239756876120512}
=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.34124872057318323, 'f1': 0.15120997062281258, 'precision': 0.17698321271287676, 'recall': 0.17129948632765396, 'hamming_loss': 0.034646878198567044}
